# HW4 Completion Notes: VAE + Transfer Learning

This notebook explains the changes completed in `variationalAutoencoderFinished.py`
and `transferLearningFinished.py`

## 1) Variational Autoencoder (Forward Pass)

 implemented the following:
- `relu(x)` now returns `np.maximum(0.0, x)`
- Hidden layer: `h = relu(x @ W1 + b1)`
- Mean head: `mu = h @ W_mu + b_mu`
- Log-variance head: `logvar = h @ W_lv + b_lv`
- Reparameterization: `z = mu + exp(0.5 * logvar) * eps`



In [2]:
# Reference snippet of the completed VAE forward logic
import numpy as np

def relu(x):
    return np.maximum(0.0, x)

def vae_encoder_forward(x, W1, b1, W_mu, b_mu, W_lv, b_lv, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    h = relu(x @ W1 + b1)
    mu = h @ W_mu + b_mu
    logvar = h @ W_lv + b_lv

    eps = rng.normal(size=mu.shape)
    z = mu + np.exp(0.5 * logvar) * eps
    return mu, logvar, z

## 2) Transfer Learning with VGG16

### What was incomplete
The file had missing dataset setup, `OOO` placeholders in model construction, incomplete prediction/evaluation steps, and syntax/runtime blockers.

### What was implemented
- Added required TensorFlow/Keras and plotting imports
- Loaded CIFAR-10 via `datasets.cifar10.load_data()`
- Printed train/test image and label shapes
- Normalized images to `[0, 1]`
- One-hot encoded labels with `to_categorical`
- Fixed image display (removed invalid `.numpy()` on NumPy arrays)
- Displayed two images with different `idx` values
- Built model by replacing placeholders with `base_model` and `prediction_layer`
- Compiled, trained, and evaluated model on test data

### Why these changes matter
- The script is now end-to-end runnable for transfer learning classification.
- Data format and loss function now match (`categorical_crossentropy` + one-hot labels).
- Test-set accuracy reporting is now included.

In [3]:
# Reference snippet of the completed transfer-learning pipeline
from tensorflow.keras import datasets, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

X_train = train_images.astype('float32') / 255.0
X_test = test_images.astype('float32') / 255.0
Y_train = to_categorical(train_labels, num_classes=10)
Y_test = to_categorical(test_labels, num_classes=10)

base_model = VGG16(weights='imagenet', include_top=False, input_shape=X_train[0].shape)
base_model.trainable = False

model = models.Sequential([
    base_model,
    Flatten(),
    Dense(50, activation='relu'),
    Dense(20, activation='relu'),
    Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
es = EarlyStopping(monitor='val_accuracy', mode='max', patience=5, restore_best_weights=True)
# model.fit(X_train, Y_train, epochs=10, validation_split=0.2, batch_size=32, callbacks=[es])
# test_loss, test_acc = model.evaluate(X_test, Y_test, verbose=0)

/Users/madams1/Documents/FAU/boilerplate/.venv/lib/python3.9/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/madams1/Documents/FAU/boilerplate/.venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/madams1/Documents/FAU/boilerplate/.venv/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9

Exception: URL fetch failure on https://storage.googleapis.com/tensorflow/keras-applications/vgg16/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5: None -- [Errno 8] nodename nor servname provided, or not known

## Quick Summary

- Both files were fully completed from placeholders to runnable code.
- VAE file now has a correct forward-pass + reparameterization implementation.
- Transfer-learning file now has complete data prep, model assembly, training, and evaluation flow.

